In [1]:
%pip install osmium pyarrow networkx scipy requests pandas h3 folium

In [2]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_29_nationale_5min_osm.py"
)

print("Script trouvé :", script.exists())

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)

national_walk.run("preflight")


In [4]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_29_nationale_5min_osm_CORRIGE.py"
)

print("Script corrigé trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError("Importer le script corrigé.")

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)

national_walk.run("prepare_inputs")


In [5]:

# ETAPE 29.2 — EXTRACTION OSM NATIONALE

national_walk.run("extract_osm")


In [6]:

# PIPELINE NATIONAL - EXECUTION AUTOMATIQUE

etapes = [
    "prepare_edges",
    "compute",
    "publish",
    "maps"
]

for etape in etapes:
    print(f"\n{'=' * 50}")
    print(f"DEMARRAGE : {etape.upper()}")
    print(f"{'=' * 50}")

    national_walk.run(etape)

    print(f"TERMINE : {etape.upper()}")

print("\nPIPELINE NATIONAL TERMINE !")


In [7]:

from pathlib import Path

base = Path("/lakehouse/default/Files")

edges = base / "silver/osm_walk_national_v1/edges_tiles"
results = base / "gold/osm_walk_national_v1/results"

# Les 24 régions définies dans le script
regions_ok = [
    region for region in REGIONS
    if (edges / f"region={region}" / "_SUCCESS").exists()
]

regions_manquantes = [
    region for region in REGIONS
    if region not in regions_ok
]

# Les 32 lots de calcul piéton
lots_ok = [
    i for i in range(32)
    if (results / f"bucket_id={i}" / "_SUCCESS").exists()
]

print("=== CONTROLE PIPELINE NATIONAL ===")
print("Régions préparées :", len(regions_ok), "/ 24")
print("Régions manquantes :", regions_manquantes)
print("Lots calculés :", len(lots_ok), "/ 32")

print("\n=== TABLES FINALES ===")
for table in [
    "gold.france_access_walk_5min_national_200m",
    "gold.france_access_walk_5min_national_bilan"
]:
    print(table, ":", spark.catalog.tableExists(table))


In [8]:

from notebookutils import mssparkutils

base = "Files/silver/osm_walk_national_v1/edges_tiles"

print("=== VERIFICATION REELLE DES FICHIERS OSM ===")

try:
    dossiers = mssparkutils.fs.ls(base)

    regions = sorted([
        f.name.rstrip("/").replace("region=", "")
        for f in dossiers
        if f.name.startswith("region=")
    ])

    print("Dossiers regionaux presents :", len(regions))
    print("Regions presentes :", regions)

except Exception as e:
    print("Impossible de lister le dossier :", str(e)[:500])
    regions = []

print("\n=== CONTROLE DE 3 REGIONS ===")

for region in ["alsace", "martinique", "reunion"]:
    chemin = f"{base}/region={region}"

    try:
        fichiers = mssparkutils.fs.ls(chemin)

        buckets = [
            f for f in fichiers
            if f.name.startswith("bucket_id=")
        ]

        marqueur = any(
            f.name == "_SUCCESS"
            for f in fichiers
        )

        print(f"\n{region}")
        print("  Dossier accessible : OUI")
        print("  Partitions presentes :", len(buckets))
        print("  Marqueur _SUCCESS :", marqueur)

        if buckets:
            exemples = mssparkutils.fs.ls(buckets[0].path)

            print(
                "  Fichiers Parquet dans la 1ere partition :",
                sum(
                    f.name.endswith(".parquet")
                    for f in exemples
                )
            )

    except Exception as e:
        print(f"\n{region} : dossier inaccessible")
        print(str(e)[:300])


In [10]:

from pathlib import Path
from notebookutils import mssparkutils

# Charger la version corrigée
script = Path(
    "/lakehouse/default/Files/"
    "etape_29_nationale_5min_osm_ONELAKE_FIX.py"
)

print("Script corrigé trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError("Importer le script corrigé")

exec(compile(
    script.read_text(encoding="utf-8"),
    str(script),
    "exec"
))

# Vérifier les vraies données dans OneLake
base = "Files/silver/osm_walk_national_v1"

regions_manquantes = [
    r for r in REGIONS
    if not mssparkutils.fs.exists(
        f"{base}/edges_tiles/region={r}/_SUCCESS"
    )
]

print("\n=== VERIFICATION ONELAKE ===")
print("Régions validées :", 24 - len(regions_manquantes), "/ 24")
print("Régions manquantes :", regions_manquantes)

print(
    "Population préparée :",
    mssparkutils.fs.exists(
        f"{base}/population_tiles/_SUCCESS"
    )
)

print(
    "Offres préparées :",
    mssparkutils.fs.exists(
        f"{base}/offers_tiles/_SUCCESS"
    )
)

# Lancer uniquement si tout est prêt
if not regions_manquantes:
    print("\n=== DEMARRAGE DU CALCUL NATIONAL ===")
    national_walk.run("compute")
else:
    print("Calcul non lancé : régions à vérifier.")


In [11]:

print("=== PUBLICATION DES RESULTATS NATIONAUX ===")

national_walk.run("publish")

print("\n=== VALIDATION DU CALCUL ===")

table = "gold.france_access_walk_5min_national_200m"

nb_carreaux = spark.table(table).count()
nb_attendus = 2324577

print("Carreaux attendus :", nb_attendus)
print("Carreaux obtenus :", nb_carreaux)
print("Difference :", nb_attendus - nb_carreaux)

print("\n=== BILAN NATIONAL 5 MIN PIETON ===")

display(
    spark.table(
        "gold.france_access_walk_5min_national_bilan"
    )
)

if nb_carreaux != nb_attendus:
    raise RuntimeError(
        "Population incomplete : verifier avant "
        "de generer les cartes."
    )

print("\n=== GENERATION DES CARTES ===")

national_walk.run("maps")

print("\n=== PIPELINE NATIONAL TERMINE ===")


In [2]:
%pip install h3 folium branca pandas pyarrow

In [4]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_30_carte_zoom_adaptatif_5min_CORRIGE.py"
)

print("Script corrigé trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError(
        "Importe le nouveau script dans Files."
    )

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)


In [1]:
%pip install networkx scipy pandas pyarrow

In [2]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_31_population_desservie_par_station_5min.py"
)

print("Script trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError(
        "Importe le script à la racine du dossier Files."
    )

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)

# Calcul par point de mobilité
station_service.run("compute")

# Publication des résultats en table Gold
station_service.run("publish")

# Ajout des arrêts et stations à la carte multi-zoom
station_service.run("maps")


In [3]:

from pyspark.sql import functions as F

df = spark.table(
    "gold.france_stations_arrets_5min_population_desservie"
)

print("=== TOP 10 STATIONS CITIZ ===")

display(
    df.filter(
        (F.col("categorie") == "citiz_avec_vehicule")
        & (F.col("statut_calcul") == "evaluable")
    )
    .select(
        "nom",
        "operateur",
        "population_desservie_5min",
        "nb_carreaux_desservis"
    )
    .orderBy(F.desc("population_desservie_5min"))
    .limit(10)
)

print("=== CONTROLE DES METADONNEES GTFS ===")

display(
    df.filter(F.col("categorie") == "arret_gtfs")
    .agg(
        F.count("*").alias("nb_arrets"),
        F.sum(
            F.when(
                F.length("operateur") > 200, 1
            ).otherwise(0)
        ).alias("operateur_anormalement_long")
    )
)


In [4]:

from pyspark.sql import functions as F

source = spark.table(
    "gold.france_stations_arrets_5min_population_desservie"
)

# Conserver le champ original pour traçabilité.
# Créer un champ fiable pour l'affichage.
clean = (
    source
    .withColumn(
        "qualite_operateur",
        F.when(
            (F.col("categorie") == "arret_gtfs")
            & (F.length(F.col("operateur")) > 200),
            F.lit("a_verifier")
        ).otherwise(F.lit("non_signale"))
    )
    .withColumn(
        "operateur_affiche",
        F.when(
            F.col("qualite_operateur") == "a_verifier",
            F.lit("Reseau GTFS - operateur a verifier")
        ).otherwise(F.col("operateur"))
    )
)

table = (
    "gold.france_stations_arrets_5min_"
    "population_desservie_clean"
)

(
    clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table)
)

print("Table nettoyee :", table)

print("\n=== CONTROLE QUALITE ===")

display(
    spark.table(table)
    .groupBy("categorie", "qualite_operateur")
    .count()
    .orderBy("categorie", "qualite_operateur")
)

print("\n=== EXEMPLES GTFS A VERIFIER ===")

display(
    spark.table(table)
    .filter(F.col("qualite_operateur") == "a_verifier")
    .select(
        "nom",
        "operateur_affiche",
        "population_desservie_5min"
    )
    .limit(10)
)


In [5]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_31B_carte_stations_operateurs_controles.py"
)

print("Script trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError("Importer le script dans Files")

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)


In [6]:

from pathlib import Path
import re

dossier = Path("/lakehouse/default/Files/gold/maps")

territoires = ["metropole", "martinique", "reunion"]

# Repérer le script des stations exécuté trop tôt
pattern = re.compile(
    r"<script>\s*(?=\(function\(\)\{\s*"
    r"'use strict';\s*const map=window\[)",
    re.S
)

for territoire in territoires:

    source = dossier / (
        f"mobilite_5min_zoom_avec_stations_qualite_{territoire}.html"
    )

    html = source.read_text(encoding="utf-8")
    match = pattern.search(html)

    if not match:
        print(f"{territoire} : script à corriger introuvable")
        continue

    # Extraire le JavaScript des stations
    fin = html.find("</script>", match.end())

    if fin == -1:
        raise RuntimeError("Fin du script introuvable")

    js = html[match.end():fin].strip()

    # Supprimer l'ancien script placé trop tôt
    html = (
        html[:match.start()]
        + html[fin + len("</script>"):]
    )

    # Exécuter le code APRÈS la création de Leaflet
    position = html.rfind("</script>")

    if position <= html.rfind("</body>"):
        raise RuntimeError(
            "Impossible de localiser le script Leaflet final"
        )

    html = (
        html[:position]
        + "\n" + js + "\n"
        + html[position:]
    )

    destination = dossier / (
        f"mobilite_5min_stations_FIX_{territoire}.html"
    )

    destination.write_text(html, encoding="utf-8")

    print(
        f"{territoire.upper()} : CORRIGE | "
        f"{destination.stat().st_size / 1024**2:.2f} Mo"
    )

print("\n=== CARTES CORRIGEES ===")
